# 01 — Residue Classes mod 6

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** first measurable constraint result.

This notebook tests the first direct prime-number constraint:

\[
p > 3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

The measurement is intentionally simple:

> Prime residues greater than 3 remain under constraint in \(1,5 \pmod{6}\).  
> Invalid residue classes drift out by divisibility.

## 0. Setup

This notebook follows the `prime-numbers-lab` measurement template:

1. define one constraint  
2. generate prime data  
3. measure what remains under constraint  
4. measure drift  
5. export figures, data, docs, TeX, and a zip bundle

In [ ]:
# Standard library
from pathlib import Path
import json
import math
import zipfile

# Data / compute
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Optional SymPy support
try:
    import sympy as sp
    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False

NOTEBOOK_ID = "01_residue_classes_mod6"
NOTEBOOK_TITLE = "Residue Classes mod 6"
REPO_NAME = "prime-numbers-lab"

OUT = Path("outputs") / NOTEBOOK_ID
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")
print(f"SymPy available: {HAS_SYMPY}")

## 1. Premise

Notebook 00 defines the orientation language. This notebook uses it directly.

- **Remains under constraint / persists:** structure retained under filtering.
- **Drift:** measurable deviation from constrained structure.
- **Recoverability:** ability to reconstruct structure from partial observations.

Here, modulo 6 provides a necessary constraint for primes greater than 3.

## 2. Constraint definition

Every integer has one residue modulo 6:

\[
n \equiv 0,1,2,3,4,5 \pmod{6}
\]

For primes greater than 3, only two residues remain possible:

\[
p > 3 \Rightarrow p \equiv 1 \text{ or } 5 \pmod{6}
\]

Reason:

- \(0,2,4 \pmod{6}\) are even.
- \(3 \pmod{6}\) is divisible by 3.
- only \(1,5 \pmod{6}\) remain as possible prime residue classes.

This does **not** mean every \(1\) or \(5 \pmod{6}\) number is prime.  
It means every prime greater than 3 satisfies the residue constraint.

In [ ]:
# Parameters

N_MAX = 1_000_000
RANDOM_SEED = 9423

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "constraint": "p > 3 implies p mod 6 in {1,5}",
}

params

## 3. Data generation

Generate primes below \(N_{\max}\).

The notebook uses SymPy if available and otherwise falls back to a simple sieve.

In [ ]:
def generate_primes(n_max: int) -> np.ndarray:
    """Generate primes less than n_max.

    Uses SymPy when available; otherwise falls back to a simple sieve.
    """
    if n_max < 2:
        return np.array([], dtype=int)

    if HAS_SYMPY:
        return np.array(list(sp.primerange(2, n_max)), dtype=int)

    sieve = np.ones(n_max, dtype=bool)
    sieve[:2] = False
    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max:i] = False
    return np.nonzero(sieve)[0].astype(int)

primes = generate_primes(N_MAX)
primes_gt3 = primes[primes > 3]
integers = np.arange(1, N_MAX)

summary = {
    "n_max": int(N_MAX),
    "integer_count_1_to_nmax_minus_1": int(len(integers)),
    "prime_count": int(len(primes)),
    "prime_count_gt3": int(len(primes_gt3)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
}

summary

## 4. Measurement: residues modulo 6

Measure two distributions:

1. all integers modulo 6  
2. primes greater than 3 modulo 6  

Then compare them.

In [ ]:
def residue_counts(values: np.ndarray, modulus: int = 6) -> pd.DataFrame:
    residues = values % modulus
    counts = np.bincount(residues, minlength=modulus)
    total = counts.sum()
    return pd.DataFrame({
        "residue_mod_6": np.arange(modulus),
        "count": counts.astype(int),
        "share": counts / total if total else np.zeros(modulus),
    })

integer_residue_df = residue_counts(integers, 6)
prime_residue_df = residue_counts(primes_gt3, 6)

comparison_df = integer_residue_df.merge(
    prime_residue_df,
    on="residue_mod_6",
    suffixes=("_integers", "_primes_gt3")
)

comparison_df

## 5. CGCS score and drift

For this notebook:

\[
CGCS_{mod6} =
\frac{\#\{p>3 : p \equiv 1,5 \pmod{6}\}}
{\#\{p>3\}}
\]

Drift is the invalid residue share:

\[
drift_{mod6} = 1 - CGCS_{mod6}
\]

Expected result:

\[
CGCS_{mod6} = 1,\quad drift_{mod6}=0
\]

In [ ]:
valid_residues = {1, 5}

prime_residues = primes_gt3 % 6
valid_mask = np.isin(prime_residues, list(valid_residues))

valid_prime_count = int(valid_mask.sum())
invalid_prime_count = int((~valid_mask).sum())
total_primes_gt3 = int(len(primes_gt3))

cgcs_mod6 = valid_prime_count / total_primes_gt3 if total_primes_gt3 else float("nan")
drift_mod6 = 1.0 - cgcs_mod6

measurement = {
    "valid_prime_count_mod6_1_or_5": valid_prime_count,
    "invalid_prime_count_mod6_not_1_or_5": invalid_prime_count,
    "total_primes_gt3": total_primes_gt3,
    "cgcs_mod6": float(cgcs_mod6),
    "drift_mod6": float(drift_mod6),
}

cgcs = {
    "score": float(cgcs_mod6),
    "definition": "CGCS_mod6 = #{p>3: p mod 6 in {1,5}} / #{p>3}",
    "interpretation": "1.0 means all primes greater than 3 remain under the mod 6 residue constraint.",
}

measurement

## 6. Recoverability note

Modulo 6 does not recover primes exactly.

It recovers **candidate residue classes**.

That distinction is important:

\[
p>3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

but

\[
n \equiv \pm 1 \pmod{6} \nRightarrow n \text{ is prime}
\]

So this notebook measures a **necessary constraint**, not a sufficient prime test.

In [ ]:
# Candidate recovery check: integers in valid residues include primes and composites.

candidate_mask = np.isin(integers % 6, list(valid_residues))
candidates = integers[candidate_mask]

prime_set = set(primes.tolist())
candidate_prime_mask = np.array([n in prime_set for n in candidates])

candidate_count = int(len(candidates))
candidate_prime_count = int(candidate_prime_mask.sum())
candidate_composite_or_one_count = candidate_count - candidate_prime_count

recoverability = {
    "candidate_count_mod6_1_or_5": candidate_count,
    "candidate_prime_count": candidate_prime_count,
    "candidate_nonprime_count": candidate_composite_or_one_count,
    "prime_share_among_candidates": float(candidate_prime_count / candidate_count),
    "note": "mod 6 recovers candidate classes, not primes exactly",
}

recoverability

## 7. Visualization

The first figure compares residue distribution for all integers versus primes greater than 3.

The second figure shows the valid-residue share by increasing scale.

In [ ]:
# Figure 1: side-by-side residue distributions.

x = comparison_df["residue_mod_6"].to_numpy()
width = 0.36

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, comparison_df["share_integers"], width, label="all integers")
ax.bar(x + width/2, comparison_df["share_primes_gt3"], width, label="primes > 3")
ax.set_title("Residue distribution modulo 6")
ax.set_xlabel("residue mod 6")
ax.set_ylabel("share")
ax.set_xticks(x)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig1_path = FIG_DIR / "residue_distribution_mod6.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

In [ ]:
# Figure 2: valid residue share by scale.

scales = np.array([10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000])
scales = scales[scales <= N_MAX]

rows = []
for scale in scales:
    ps = primes[(primes > 3) & (primes < scale)]
    if len(ps) == 0:
        share = np.nan
        drift = np.nan
    else:
        share = float(np.isin(ps % 6, list(valid_residues)).mean())
        drift = 1.0 - share
    rows.append({
        "scale_x": int(scale),
        "prime_count_gt3_below_x": int(len(ps)),
        "cgcs_mod6_below_x": share,
        "drift_mod6_below_x": drift,
    })

scale_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scale_df["scale_x"], scale_df["cgcs_mod6_below_x"], marker="o", label="CGCS mod 6")
ax.plot(scale_df["scale_x"], scale_df["drift_mod6_below_x"], marker="o", label="drift mod 6")
ax.set_xscale("log")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Constraint score by scale")
ax.set_xlabel("x")
ax.set_ylabel("score")
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / "cgcs_and_drift_by_scale_mod6.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

scale_df, fig2_path

## 8. Interpretation

Use this notebook as the baseline CGCS demonstration:

1. **What remains under constraint?**  
   Primes greater than 3 remain in residues \(1,5 \pmod{6}\).

2. **What drifts?**  
   Invalid residues \(0,2,3,4 \pmod{6}\) drift out of the prime set because they are divisible by 2 or 3.

3. **What is recoverable?**  
   The residue candidate classes are recoverable. Prime identity is not fully recoverable from mod 6 alone.

4. **What should not be overclaimed?**  
   Modulo 6 is necessary but not sufficient for primality.

In [ ]:
interpretation = f'''
# {NOTEBOOK_TITLE}

## Constraint result

This notebook tested the residue constraint

p > 3 => p mod 6 in {{1,5}}.

For primes below {N_MAX:,}, the measured score was:

- CGCS_mod6 = {cgcs_mod6:.6f}
- drift_mod6 = {drift_mod6:.6f}

## Remains under constraint

Every prime greater than 3 remained in residue classes 1 or 5 modulo 6.

## Drift

Invalid residue classes 0, 2, 3, and 4 modulo 6 had zero representation among primes greater than 3. They drift out by divisibility.

## Recoverability

Modulo 6 recovers prime candidate classes, not primes exactly. Many integers congruent to 1 or 5 modulo 6 are composite.

## Caution

This notebook does not prove RH and does not provide a sufficient primality test. It provides the first direct measurement of structure remaining under a finite residue constraint.
'''.strip()

print(interpretation)

## 9. Export data, notes, math, and TeX

This block writes:

- `summary.csv`
- `residue_counts.csv`
- `scale_scores.csv`
- `metadata.json`
- `interpretation.md`
- `design_notes.md`
- `summary_snippet.tex`
- `math_notes.tex`

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **measurement,
    **recoverability,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

csv_summary_path = DATA_DIR / "summary.csv"
csv_residue_path = DATA_DIR / "residue_counts.csv"
csv_scale_path = DATA_DIR / "scale_scores.csv"
json_path = DATA_DIR / "metadata.json"

interpretation_md_path = DOCS_DIR / "interpretation.md"
design_notes_md_path = DOCS_DIR / "design_notes.md"

summary_tex_path = TEX_DIR / "summary_snippet.tex"
math_tex_path = TEX_DIR / "math_notes.tex"

summary_df.to_csv(csv_summary_path, index=False)
comparison_df.to_csv(csv_residue_path, index=False)
scale_df.to_csv(csv_scale_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "recoverability": recoverability,
    "cgcs": cgcs,
    "figures": [str(fig1_path), str(fig2_path)],
    "data": {
        "summary": str(csv_summary_path),
        "residue_counts": str(csv_residue_path),
        "scale_scores": str(csv_scale_path),
    },
    "docs": {
        "interpretation": str(interpretation_md_path),
        "design_notes": str(design_notes_md_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

json_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_md_path.write_text(interpretation + "\n", encoding="utf-8")

design_notes = f'''
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 01 is the first measurement notebook in prime-numbers-lab.

It establishes a direct finite constraint before moving into gaps, density, sieve structure, random comparisons, recoverability, or zeta bridges.

## Constraint

For primes greater than 3:

p mod 6 must be 1 or 5.

## Measurement

The notebook counts residues modulo 6 for:

1. all integers below N_MAX
2. primes greater than 3 below N_MAX

It compares residue shares and computes a direct CGCS score.

## CGCS score

CGCS_mod6 = #{{p>3: p mod 6 in {{1,5}}}} / #{{p>3}}

For this necessary residue constraint, the expected result is exactly 1.0.

## Drift

drift_mod6 = 1 - CGCS_mod6

The expected drift is exactly 0.0.

## Recoverability

Modulo 6 recovers candidate residue classes, not primes exactly. It is a necessary condition, not a sufficient condition.

## Handoff

Notebook 02 should measure prime gaps and show how ordered prime structure continues across scale.
'''.strip()

design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf'''
\section*{{{NOTEBOOK_TITLE}}}

This notebook tests the finite residue constraint
\[
p > 3 \Rightarrow p \equiv 1 \text{{ or }} 5 \pmod{{6}}.
\]

For primes below {N_MAX:,}, the measurement gives:
\begin{{itemize}}
  \item $\#\{{p>3\}} = {total_primes_gt3}$
  \item valid residue count $= {valid_prime_count}$
  \item invalid residue count $= {invalid_prime_count}$
  \item $CGCS_{{mod6}} = {cgcs_mod6:.6f}$
  \item $drift_{{mod6}} = {drift_mod6:.6f}$
\end{{itemize}}

Modulo 6 recovers candidate residue classes, not primes exactly.
'''.strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf'''
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: Residue Classes mod 6}}

\subsection*{{Residue classes}}

Every integer has one residue modulo 6:
\[
n \equiv 0,1,2,3,4,5 \pmod{{6}}.
\]

\subsection*{{Prime residue constraint}}

For every prime greater than 3:
\[
p > 3 \Rightarrow p \equiv \pm 1 \pmod{{6}}.
\]

Equivalently:
\[
p > 3 \Rightarrow p \equiv 1 \text{{ or }} 5 \pmod{{6}}.
\]

\subsection*{{Necessary but not sufficient}}

The residue constraint is necessary:
\[
p>3 \Rightarrow p \equiv \pm 1 \pmod{{6}}.
\]

It is not sufficient:
\[
n \equiv \pm 1 \pmod{{6}} \nRightarrow n \text{{ is prime}}.
\]

\subsection*{{CGCS score}}

For this notebook:
\[
CGCS_{{mod6}} =
\frac{{\#\{{p>3 : p \equiv 1,5 \pmod{{6}}\}}}}
{{\#\{{p>3\}}}}.
\]

Measured result:
\[
CGCS_{{mod6}} = {cgcs_mod6:.6f}.
\]

\subsection*{{Drift}}

\[
drift_{{mod6}} = 1 - CGCS_{{mod6}}.
\]

Measured result:
\[
drift_{{mod6}} = {drift_mod6:.6f}.
\]

\subsection*{{Recoverability}}

Modulo 6 recovers candidate residue classes:
\[
n \equiv \pm 1 \pmod{{6}}.
\]

It does not recover prime identity exactly.

\end{{document}}
'''.strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

csv_summary_path, csv_residue_path, csv_scale_path, json_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 10. Optional results bundle

Package notebook outputs for easy download or repo transfer.

In [ ]:
bundle_path = OUT / f"{NOTEBOOK_ID}_results_bundle.zip"

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for subdir in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
        for path in subdir.rglob("*"):
            if path.is_file():
                z.write(path, arcname=str(path.relative_to(OUT)))

bundle_path

## 11. Next notebook handoff

Next notebook:

```text
02_prime_gaps.ipynb
```

Purpose:

> measure how the ordered prime sequence continues across scale through gap structure, then treat large deviations as drift rather than randomness.

In [ ]:
next_step = "Notebook 02: prime gaps — ordered structure across scale."
print(next_step)